# Cumulocity → Snowflake: query examples

Read-only examples for one tenant; requires `cumulocity_build_up.ipynb` to have run for it. Set the tenant in the next cell; queries use the derived names via `{{raw_db}}` / `{{app_db}}`. To copy a query elsewhere, replace those with the real names (e.g. `CUMULOCITY_T15264971`).

| Source | What | One row per |
|---|---|---|
| `CUMULOCITY_<T>.CURATED.DEVICE_LATEST` | Current device state; all fields in `ATTRIBUTES` | device (deleted excluded) |
| `CUMULOCITY_<T>.CURATED.ALARM_LATEST` | Current alarm state (`ACTIVE` / `CLEARED`) | alarm |
| `CUMULOCITY_ICEBERG_<T>."cdc_measurement"."<series>"` | Measurements; value in an object column named like the series, e.g. `"bearingTemperature":value` | measurement |
| `CUMULOCITY_ICEBERG_<T>."cdc_event"."event"` | Events (insert-only) | event |

**Join key:** every alarm, event and measurement has `"source"` = `DEVICE_LATEST."id"`. Use `LEFT JOIN` so rows of deleted or unknown devices are kept.

Cumulocity column names are case-sensitive and must be double-quoted (`"time"`, `"source"`).

In [ ]:
import re

catalog_name = ""   # Cumulocity tenant ID, as in the build-up notebook

assert re.fullmatch(r"[A-Za-z0-9]+", catalog_name), "catalog_name must be alphanumeric (used in object names)"
tenant = catalog_name.upper()
raw_db = f"CUMULOCITY_ICEBERG_{tenant}"   # raw Iceberg tables
app_db = f"CUMULOCITY_{tenant}"           # CURATED.DEVICE_LATEST, CURATED.ALARM_LATEST
wh     = f"CUMULOCITY_WH_{tenant}"
print(raw_db, app_db, wh)

In [ ]:
%%sql -r set_role
USE ROLE ACCOUNTADMIN

In [ ]:
%%sql -r set_warehouse
USE WAREHOUSE {{wh}}

## 1. Devices

In [ ]:
%%sql -r devices_per_type
SELECT "type" AS device_type,
       COUNT(*) AS devices,
       MAX("lastUpdated") AS last_change
FROM {{app_db}}.CURATED.DEVICE_LATEST
GROUP BY 1
ORDER BY devices DESC

Fields that aren't regular columns live in `ATTRIBUTES` (an OBJECT). Access them with `:` and cast them; arrays can be flattened. Fields that are empty for a device are missing from its object and return NULL.

In [ ]:
%%sql -r device_attributes
SELECT "id", "name", "type",
       attributes:c8y_Notes::STRING              AS notes,
       attributes:parentName::STRING             AS parent_name,
       ARRAY_SIZE(attributes:childDevices)       AS child_devices,
       attributes:c8y_SupportedOperations        AS supported_operations,
       ARRAY_SIZE(OBJECT_KEYS(attributes))       AS filled_fields
FROM {{app_db}}.CURATED.DEVICE_LATEST
WHERE attributes:c8y_SupportedOperations IS NOT NULL
ORDER BY "lastUpdated" DESC
LIMIT 20

In [ ]:
%%sql -r devices_by_operation
SELECT op.value::STRING AS operation,
       COUNT(*) AS devices,
       ARRAY_AGG(d."name") WITHIN GROUP (ORDER BY d."name") AS device_names
FROM {{app_db}}.CURATED.DEVICE_LATEST d,
     LATERAL FLATTEN(INPUT => d.attributes:c8y_SupportedOperations) op
GROUP BY 1
ORDER BY devices DESC

## 2. Devices + alarms

In [ ]:
%%sql -r active_alarms
SELECT a."time", a."severity", a."type" AS alarm_type, a."text", a."count",
       a."source" AS device_id, d."name" AS device_name, d."type" AS device_type
FROM {{app_db}}.CURATED.ALARM_LATEST a
LEFT JOIN {{app_db}}.CURATED.DEVICE_LATEST d ON d."id" = a."source"
WHERE a."status" = 'ACTIVE'
ORDER BY a."time" DESC

In [ ]:
%%sql -r alarms_per_device
-- Highest open severity: CRITICAL > MAJOR > MINOR > WARNING
SELECT a."source" AS device_id,
       ANY_VALUE(d."name") AS device_name,
       COUNT_IF(a."status" = 'ACTIVE')  AS active_alarms,
       COUNT_IF(a."status" = 'CLEARED') AS cleared_alarms,
       CASE MAX(IFF(a."status" = 'ACTIVE',
                    DECODE(a."severity", 'CRITICAL', 4, 'MAJOR', 3, 'MINOR', 2, 'WARNING', 1, 0), NULL))
         WHEN 4 THEN 'CRITICAL' WHEN 3 THEN 'MAJOR' WHEN 2 THEN 'MINOR' WHEN 1 THEN 'WARNING'
       END AS highest_open_severity,
       MAX(a."time") AS last_alarm
FROM {{app_db}}.CURATED.ALARM_LATEST a
LEFT JOIN {{app_db}}.CURATED.DEVICE_LATEST d ON d."id" = a."source"
GROUP BY 1
ORDER BY active_alarms DESC, last_alarm DESC
LIMIT 50

## 3. Measurements

Each measurement series is its own table in `cdc_measurement`. The value sits in an object column with the series' name, e.g. `"bearingTemperature"` → `{"value": 49, "unit": "c"}`. Filter on `"time"` to keep queries fast.

In [ ]:
%%sql -r measurement_series
SELECT table_name AS series, row_count, last_altered
FROM {{raw_db}}.INFORMATION_SCHEMA.TABLES
WHERE table_schema = 'cdc_measurement'
ORDER BY row_count DESC

In [ ]:
%%sql -r latest_temperatures
WITH temperatures AS (
  SELECT 'bearingTemperature' AS series, "source", "time",
         "bearingTemperature":value::FLOAT AS value, "bearingTemperature":unit::STRING AS unit
  FROM {{raw_db}}."cdc_measurement"."bearingTemperature"
  WHERE "time" >= DATEADD('day', -1, CURRENT_TIMESTAMP())
  UNION ALL
  SELECT 'inflowTemperature', "source", "time",
         "inflowTemperature":value::FLOAT, "inflowTemperature":unit::STRING
  FROM {{raw_db}}."cdc_measurement"."inflowTemperature"
  WHERE "time" >= DATEADD('day', -1, CURRENT_TIMESTAMP())
)
SELECT d."name" AS device_name, t.series, t.value, t.unit, t."time" AS measured_at
FROM temperatures t
LEFT JOIN {{app_db}}.CURATED.DEVICE_LATEST d ON d."id" = t."source"
QUALIFY ROW_NUMBER() OVER (PARTITION BY t."source", t.series ORDER BY t."time" DESC) = 1
ORDER BY device_name, t.series

In [ ]:
%%sql -r bearing_temp_hourly
SELECT d."name" AS device_name,
       DATE_TRUNC('hour', m."time") AS hour,
       ROUND(AVG(m."bearingTemperature":value::FLOAT), 1) AS avg_temp,
       MIN(m."bearingTemperature":value::FLOAT) AS min_temp,
       MAX(m."bearingTemperature":value::FLOAT) AS max_temp,
       COUNT(*) AS measurements
FROM {{raw_db}}."cdc_measurement"."bearingTemperature" m
LEFT JOIN {{app_db}}.CURATED.DEVICE_LATEST d ON d."id" = m."source"
WHERE m."time" >= DATEADD('hour', -24, CURRENT_TIMESTAMP())
GROUP BY 1, 2
ORDER BY device_name, hour DESC

## 4. Events

In [ ]:
%%sql -r events_per_type
SELECT "type" AS event_type,
       COUNT(*) AS events,
       COUNT(DISTINCT "source") AS devices,
       MAX("time") AS latest
FROM {{raw_db}}."cdc_event"."event"
WHERE "time" >= DATEADD('day', -7, CURRENT_TIMESTAMP())
GROUP BY 1
ORDER BY events DESC

In [ ]:
%%sql -r latest_events
SELECT e."time", e."type" AS event_type, e."text",
       e."source" AS device_id, d."name" AS device_name, d."type" AS device_type
FROM {{raw_db}}."cdc_event"."event" e
LEFT JOIN {{app_db}}.CURATED.DEVICE_LATEST d ON d."id" = e."source"
WHERE e."time" >= DATEADD('day', -1, CURRENT_TIMESTAMP())
ORDER BY e."time" DESC
LIMIT 50

## 5. Device timeline

Alarms, events and measurements of one device in one time-ordered list. Set the device ID below (default: Pump01).

In [ ]:
device_id = "2069101927"   # DEVICE_LATEST."id"
hours_back = 2

assert device_id.isdigit(), "device_id must be numeric"
assert isinstance(hours_back, int) and hours_back > 0

In [ ]:
%%sql -r device_timeline
SELECT "time", 'ALARM' AS kind, "type" AS detail, "severity" || ' / ' || "status" || ': ' || "text" AS info
FROM {{app_db}}.CURATED.ALARM_LATEST
WHERE "source" = '{{device_id}}' AND "time" >= DATEADD('hour', -{{hours_back}}, CURRENT_TIMESTAMP())
UNION ALL
SELECT "time", 'EVENT', "type", "text"
FROM {{raw_db}}."cdc_event"."event"
WHERE "source" = '{{device_id}}' AND "time" >= DATEADD('hour', -{{hours_back}}, CURRENT_TIMESTAMP())
UNION ALL
-- Measurements reduced to one row per minute, so they don't drown out alarms and events
SELECT DATE_TRUNC('minute', "time"), 'MEASUREMENT', 'bearingTemperature',
       'avg ' || ROUND(AVG("bearingTemperature":value::FLOAT), 1) || ' ' || ANY_VALUE("bearingTemperature":unit::STRING)
FROM {{raw_db}}."cdc_measurement"."bearingTemperature"
WHERE "source" = '{{device_id}}' AND "time" >= DATEADD('hour', -{{hours_back}}, CURRENT_TIMESTAMP())
GROUP BY 1
ORDER BY "time" DESC
LIMIT 200